# ♟️ Chess Move Prediction with a Transformer
### Generative AI Final Project

**Objective**: Train a Transformer to learn `p(move | board)` — given a board configuration, generate the next best move.

**Pipeline overview**:
1. Download game data from Lichess (PGN format)
2. Encode board as 64 integers (one per square, 13 piece types)
3. Encode move as a single integer (from-square × 64 + to-square)
4. Train a Transformer with cross-entropy loss
5. Evaluate: move accuracy, legal move rate, top-k Stockfish match, centipawn loss

## 1. Install Dependencies

In [1]:
%pip install chess zstandard

# Install Stockfish engine binary (for evaluation)
import subprocess, sys, os

if not os.path.exists('/usr/games/stockfish'):
    try:
        subprocess.run(['sudo', 'apt-get', 'install', '-y', '-q', 'stockfish'],
                       check=True, capture_output=True)
        print('Stockfish installed.')
    except (subprocess.CalledProcessError, FileNotFoundError):
        print('WARNING: Could not install Stockfish automatically.'
              ' Install it manually (e.g. sudo apt-get install stockfish)'
              ' to enable Stockfish-based evaluation cells.')
else:
    print('Stockfish already available.')


In [ ]:
import os
import math
import random
import gzip
import urllib.request
from tqdm import tqdm

import chess
import chess.pgn
import chess.engine

import numpy as np
import matplotlib.pyplot as plt

import torch
import torch.nn as nn
import torch.nn.functional as F
from torch.utils.data import Dataset, DataLoader, random_split

# Reproducibility
SEED = 42
random.seed(SEED)
np.random.seed(SEED)
torch.manual_seed(SEED)

DEVICE = torch.device('cuda' if torch.cuda.is_available() else 'cpu')
print(f'Using device: {DEVICE}')

Using device: cuda


## 2. Download Dataset

We use the **Lichess open database** (lichess.org/database). Each monthly file contains millions of rated games in PGN format.

For this notebook we use a small sample file from Lichess (January 2013 — ~120k games, ~20MB compressed). For better results, download a recent month with millions of games.

> **Full dataset**: https://database.lichess.org  
> For production training, download e.g. `lichess_db_standard_rated_2024-01.pgn.zst` (~30GB, ~120M positions)

In [ ]:
# -----------------------------------------------------------------------
# Using the Lichess 2026-03 database (28 GB compressed .zst)
# Stream-decompressed on the fly — no disk space needed beyond the .zst
# -----------------------------------------------------------------------
PGN_FILE = '/home/zubair/Downloads/lichess_db_standard_rated_2026-03.pgn.zst'

import os
assert os.path.exists(PGN_FILE), f'File not found: {PGN_FILE}'
print(f'PGN file: {PGN_FILE}  ({os.path.getsize(PGN_FILE)/1e9:.1f} GB compressed)')


Mounted at /content/drive


## 3. Encoding: Board → 64 Integers, Move → 1 Integer

In [ ]:
# -------------------------------------------------------------------
# Piece vocabulary: 13 tokens
#   0  = empty square
#   1-6  = White P, N, B, R, Q, K
#   7-12 = Black p, n, b, r, q, k
# -------------------------------------------------------------------
PIECE_TO_IDX = {
    None: 0,
    (chess.PAWN,   chess.WHITE): 1,
    (chess.KNIGHT, chess.WHITE): 2,
    (chess.BISHOP, chess.WHITE): 3,
    (chess.ROOK,   chess.WHITE): 4,
    (chess.QUEEN,  chess.WHITE): 5,
    (chess.KING,   chess.WHITE): 6,
    (chess.PAWN,   chess.BLACK): 7,
    (chess.KNIGHT, chess.BLACK): 8,
    (chess.BISHOP, chess.BLACK): 9,
    (chess.ROOK,   chess.BLACK): 10,
    (chess.QUEEN,  chess.BLACK): 11,
    (chess.KING,   chess.BLACK): 12,
}

BOARD_VOCAB_SIZE = 13   # 0-12
MOVE_VOCAB_SIZE  = 4096 # 64 * 64 (from_sq * 64 + to_sq)
                        # promotions simplified: we ignore promotion piece


def encode_board(board: chess.Board) -> list:
    """
    Returns a list of 64 integers representing each square a1..h8.
    Squares are iterated in python-chess order: a1=0, b1=1, ..., h8=63.
    """
    tokens = []
    for sq in chess.SQUARES:  # a1=0 .. h8=63
        piece = board.piece_at(sq)
        if piece is None:
            tokens.append(0)
        else:
            tokens.append(PIECE_TO_IDX[(piece.piece_type, piece.color)])
    return tokens  # length 64


def encode_move(move: chess.Move) -> int:
    """
    Encodes move as: from_square * 64 + to_square.
    Range: [0, 4095]
    """
    return move.from_square * 64 + move.to_square


def decode_move(move_idx: int) -> chess.Move:
    """Inverse of encode_move."""
    from_sq = move_idx // 64
    to_sq   = move_idx %  64
    return chess.Move(from_sq, to_sq)


# Quick sanity check
board = chess.Board()
tokens = encode_board(board)
print(f'Board encoded: {len(tokens)} tokens, first 16: {tokens[:16]}')

sample_move = chess.Move.from_uci('e2e4')
idx = encode_move(sample_move)
print(f'Move e2e4 -> index {idx}')
print(f'Index {idx} -> {decode_move(idx).uci()}')

Board encoded: 64 tokens, first 16: [4, 2, 3, 5, 6, 3, 2, 4, 1, 1, 1, 1, 1, 1, 1, 1]
Move e2e4 -> index 796
Index 796 -> e2e4


## 4. Parse PGN and Build Dataset

In [ ]:
import io, contextlib
import zstandard as zstd


@contextlib.contextmanager
def open_pgn(path: str):
    """Open a plain .pgn or compressed .pgn.zst file as a text stream."""
    if path.endswith('.zst'):
        fh = open(path, 'rb')
        stream = zstd.ZstdDecompressor().stream_reader(fh)
        text = io.TextIOWrapper(stream, encoding='utf-8', errors='replace')
        try:
            yield text
        finally:
            fh.close()
    else:
        with open(path, 'r', errors='replace') as f:
            yield f


def parse_pgn(pgn_path: str,
              max_games: int = 100_000,
              min_elo: int = 2000) -> tuple:
    """
    Parse a PGN (or .pgn.zst) file and extract (board_tokens, move_idx) pairs.

    Args:
        pgn_path  : path to .pgn or .pgn.zst file
        max_games : stop after this many qualifying games
        min_elo   : skip games where either player is below this Elo

    Returns:
        boards : np.array of shape (N, 64), dtype int8
        moves  : np.array of shape (N,),   dtype int16
    """
    boards_list = []
    moves_list  = []
    games_parsed = 0

    with open_pgn(pgn_path) as f:
        pbar = tqdm(total=max_games, desc='Parsing games')
        while games_parsed < max_games:
            game = chess.pgn.read_game(f)
            if game is None:
                break

            # Filter by Elo
            try:
                white_elo = int(game.headers.get('WhiteElo', 0))
                black_elo = int(game.headers.get('BlackElo', 0))
                if white_elo < min_elo or black_elo < min_elo:
                    continue
            except (ValueError, TypeError):
                continue

            board = game.board()
            for move in game.mainline_moves():
                boards_list.append(encode_board(board))
                moves_list.append(encode_move(move))
                board.push(move)

            games_parsed += 1
            pbar.update(1)
        pbar.close()

    print(f'Games parsed : {games_parsed:,}')
    print(f'Positions    : {len(boards_list):,}')

    boards = np.array(boards_list, dtype=np.int8)
    moves  = np.array(moves_list,  dtype=np.int16)
    return boards, moves


# Small sample — just enough to build val_ds for evaluation.
# Full training used 500k max_games / min_elo=2000; checkpoint is already saved.
boards_np, moves_np = parse_pgn(PGN_FILE, max_games=5_000, min_elo=0)
print(f'\nBoards shape : {boards_np.shape}')
print(f'Moves  shape : {moves_np.shape}')


Parsing games: 100%|██████████| 50000/50000 [02:23<00:00, 348.16it/s]


Games parsed : 50,000
Positions    : 3,562,361

Boards shape : (3562361, 64)
Moves  shape : (3562361,)


In [ ]:
class ChessDataset(Dataset):
    def __init__(self, boards: np.ndarray, moves: np.ndarray):
        self.boards = torch.tensor(boards, dtype=torch.long)
        self.moves  = torch.tensor(moves,  dtype=torch.long)

    def __len__(self):
        return len(self.moves)

    def __getitem__(self, idx):
        return self.boards[idx], self.moves[idx]


dataset = ChessDataset(boards_np, moves_np)

# Free the numpy arrays — tensors are already copied into the dataset
import gc
del boards_np, moves_np
gc.collect()
print(f'Dataset size: {len(dataset):,} positions')

# 90/10 train/val split
val_size   = int(0.10 * len(dataset))
train_size = len(dataset) - val_size
train_ds, val_ds = random_split(dataset, [train_size, val_size],
                                generator=torch.Generator().manual_seed(SEED))

BATCH_SIZE = 1024
train_loader = DataLoader(train_ds, batch_size=BATCH_SIZE, shuffle=True,
                          num_workers=4, pin_memory=True, persistent_workers=True)
val_loader   = DataLoader(val_ds,   batch_size=BATCH_SIZE, shuffle=False,
                          num_workers=4, pin_memory=True, persistent_workers=True)

print(f'Train samples : {train_size:,}')
print(f'Val   samples : {val_size:,}')
print(f'Train batches : {len(train_loader):,}')


Train samples : 3,206,125
Val   samples : 356,236
Train batches : 6,262


## 5. Transformer Model

Architecture:
- **Embedding**: each of the 64 squares → learned embedding vector
- **Positional encoding**: sine/cosine to encode square positions
- **Transformer encoder**: standard multi-head self-attention layers
- **Classification head**: mean-pool sequence → linear → 4096 logits (one per possible move)

In [ ]:
class PositionalEncoding(nn.Module):
    """Standard sine/cosine positional encoding for sequence of length 64."""
    def __init__(self, d_model: int, max_len: int = 64, dropout: float = 0.1):
        super().__init__()
        self.dropout = nn.Dropout(dropout)

        pe = torch.zeros(max_len, d_model)
        position = torch.arange(max_len).unsqueeze(1).float()
        div_term = torch.exp(torch.arange(0, d_model, 2).float() *
                             (-math.log(10000.0) / d_model))
        pe[:, 0::2] = torch.sin(position * div_term)
        pe[:, 1::2] = torch.cos(position * div_term)
        pe = pe.unsqueeze(0)  # (1, max_len, d_model)
        self.register_buffer('pe', pe)

    def forward(self, x: torch.Tensor) -> torch.Tensor:
        # x: (batch, seq_len, d_model)
        x = x + self.pe[:, :x.size(1)]
        return self.dropout(x)


class ChessTransformer(nn.Module):
    """
    Transformer encoder for chess move prediction.

    Input  : board token sequence of shape (batch, 64)
    Output : logits over 4096 possible moves, shape (batch, 4096)
    """
    def __init__(
        self,
        board_vocab_size: int = 13,
        move_vocab_size:  int = 4096,
        d_model:          int = 256,
        nhead:            int = 8,
        num_layers:       int = 6,
        dim_feedforward:  int = 1024,
        dropout:          float = 0.1,
    ):
        super().__init__()
        self.embedding = nn.Embedding(board_vocab_size, d_model, padding_idx=0)
        self.pos_enc   = PositionalEncoding(d_model, max_len=64, dropout=dropout)

        encoder_layer = nn.TransformerEncoderLayer(
            d_model=d_model,
            nhead=nhead,
            dim_feedforward=dim_feedforward,
            dropout=dropout,
            batch_first=True,   # (batch, seq, feature)
        )
        self.transformer = nn.TransformerEncoder(encoder_layer, num_layers=num_layers)
        self.head = nn.Linear(d_model, move_vocab_size)

    def forward(self, board_tokens: torch.Tensor) -> torch.Tensor:
        # board_tokens: (batch, 64)
        x = self.embedding(board_tokens)    # (batch, 64, d_model)
        x = self.pos_enc(x)                 # (batch, 64, d_model)
        x = self.transformer(x)             # (batch, 64, d_model)
        x = x.mean(dim=1)                   # (batch, d_model)  — mean pooling
        logits = self.head(x)               # (batch, 4096)
        return logits


model = ChessTransformer(
    board_vocab_size = BOARD_VOCAB_SIZE,
    move_vocab_size  = MOVE_VOCAB_SIZE,
    d_model          = 256,
    nhead            = 8,
    num_layers       = 6,
    dim_feedforward  = 1024,
    dropout          = 0.15,
).to(DEVICE)


total_params = sum(p.numel() for p in model.parameters() if p.requires_grad)
print(f'Model parameters: {total_params:,}')
print(model)

Model parameters: 5,794,560
ChessTransformer(
  (embedding): Embedding(13, 256, padding_idx=0)
  (pos_enc): PositionalEncoding(
    (dropout): Dropout(p=0.1, inplace=False)
  )
  (transformer): TransformerEncoder(
    (layers): ModuleList(
      (0-5): 6 x TransformerEncoderLayer(
        (self_attn): MultiheadAttention(
          (out_proj): NonDynamicallyQuantizableLinear(in_features=256, out_features=256, bias=True)
        )
        (linear1): Linear(in_features=256, out_features=1024, bias=True)
        (dropout): Dropout(p=0.1, inplace=False)
        (linear2): Linear(in_features=1024, out_features=256, bias=True)
        (norm1): LayerNorm((256,), eps=1e-05, elementwise_affine=True)
        (norm2): LayerNorm((256,), eps=1e-05, elementwise_affine=True)
        (dropout1): Dropout(p=0.1, inplace=False)
        (dropout2): Dropout(p=0.1, inplace=False)
      )
    )
  )
  (head): Linear(in_features=256, out_features=4096, bias=True)
)


## 6. Training

In [ ]:
import pathlib as _pl

if _pl.Path('chess_transformer_best.pt').exists():
    print('Checkpoint already exists — skipping training.')
    print('Delete chess_transformer_best.pt to retrain from scratch.')
else:
    # ----- Hyperparameters -----
    NUM_EPOCHS    = 15
    LEARNING_RATE = 3e-4
    WEIGHT_DECAY  = 1e-4

    criterion = nn.CrossEntropyLoss()
    optimizer = torch.optim.AdamW(model.parameters(), lr=LEARNING_RATE, weight_decay=WEIGHT_DECAY)
    scheduler = torch.optim.lr_scheduler.CosineAnnealingLR(optimizer, T_max=NUM_EPOCHS)

    def run_epoch(loader, training: bool):
        model.train(training)
        total_loss, total_correct, total_samples = 0.0, 0, 0

        with torch.set_grad_enabled(training):
            for boards, moves in loader:
                boards = boards.to(DEVICE)
                moves  = moves.to(DEVICE)

                logits = model(boards)
                loss   = criterion(logits, moves)

                if training:
                    optimizer.zero_grad()
                    loss.backward()
                    torch.nn.utils.clip_grad_norm_(model.parameters(), 1.0)
                    optimizer.step()

                preds = logits.argmax(dim=1)
                total_correct  += (preds == moves).sum().item()
                total_loss     += loss.item() * boards.size(0)
                total_samples  += boards.size(0)

        return total_loss / total_samples, total_correct / total_samples

    history = {'train_loss': [], 'val_loss': [], 'train_acc': [], 'val_acc': []}
    best_val_loss = float('inf')

    for epoch in range(1, NUM_EPOCHS + 1):
        train_loss, train_acc = run_epoch(train_loader, training=True)
        val_loss,   val_acc   = run_epoch(val_loader,   training=False)
        scheduler.step()

        history['train_loss'].append(train_loss)
        history['val_loss'].append(val_loss)
        history['train_acc'].append(train_acc)
        history['val_acc'].append(val_acc)

        if val_loss < best_val_loss:
            best_val_loss = val_loss
            torch.save(model.state_dict(), 'chess_transformer_best.pt')

        print(f'Epoch {epoch:02d}/{NUM_EPOCHS} | '
              f'Train Loss: {train_loss:.4f}  Acc: {train_acc*100:.2f}% | '
              f'Val Loss: {val_loss:.4f}  Acc: {val_acc*100:.2f}%')

    print(f'\nBest val loss: {best_val_loss:.4f}')
    print('Checkpoint saved: chess_transformer_best.pt')

    import json
    _pl.Path('training_history.json').write_text(json.dumps({
        'history': history,
        'train_size': train_size,
        'val_size': val_size,
        'num_epochs': NUM_EPOCHS,
    }))
    print('Training history saved to training_history.json')


In [ ]:
if not history['train_loss']:
    print('No training history available — skipping loss/accuracy plots.')
else:
    epochs = range(1, NUM_EPOCHS + 1)
    fig, axes = plt.subplots(1, 2, figsize=(13, 4))

    axes[0].plot(epochs, history['train_loss'], label='Train Loss', marker='o')
    axes[0].plot(epochs, history['val_loss'],   label='Val Loss',   marker='s')
    axes[0].set_title('Cross-Entropy Loss')
    axes[0].set_xlabel('Epoch')
    axes[0].set_ylabel('Loss')
    axes[0].legend()
    axes[0].grid(True)

    axes[1].plot(epochs, [a*100 for a in history['train_acc']], label='Train Acc', marker='o')
    axes[1].plot(epochs, [a*100 for a in history['val_acc']],   label='Val Acc',   marker='s')
    axes[1].set_title('Move Accuracy (%)')
    axes[1].set_xlabel('Epoch')
    axes[1].set_ylabel('Accuracy (%)')
    axes[1].legend()
    axes[1].grid(True)

    plt.tight_layout()
    plt.savefig('training_curves.png', dpi=150)
    plt.show()


## 7. Evaluation

We evaluate on a held-out test set using four metrics:

| Metric | What it measures |
|---|---|
| **Move accuracy** | % of predicted moves matching ground truth |
| **Legal move rate** | % of predicted moves that are legal in the position |
| **Top-k match rate** | % of predicted moves in Stockfish's top-k |
| **Centipawn loss** | Average centipawn difference vs Stockfish best move |

In [ ]:
import json, pathlib

# Load the best checkpoint
model.load_state_dict(torch.load('chess_transformer_best.pt',
                                   map_location=DEVICE,
                                   weights_only=False))
model.eval()
print('Loaded best model checkpoint.')

# Restore training history and dataset sizes saved after training
_hist_path = pathlib.Path('training_history.json')
if _hist_path.exists():
    _saved     = json.loads(_hist_path.read_text())
    history    = _saved['history']
    train_size = _saved.get('train_size', train_size)
    val_size   = _saved.get('val_size',   val_size)
    NUM_EPOCHS = _saved.get('num_epochs', len(history['train_loss']))
    print(f'Loaded training history: {NUM_EPOCHS} epochs, '
          f'{train_size:,} train / {val_size:,} val positions.')
else:
    history    = {'train_loss': [], 'val_loss': [], 'train_acc': [], 'val_acc': []}
    NUM_EPOCHS = 0
    print('No training_history.json found — run the training cell once to generate it.')


In [ ]:
# -------------------------------------------------------------------
# Metric 1 & 2: Move Accuracy + Legal Move Rate
# Run on the full validation set — fast, no Stockfish needed
# -------------------------------------------------------------------

def predict_move(board_tokens: torch.Tensor, board: chess.Board = None,
                 enforce_legal: bool = False) -> chess.Move:
    """
    Run the model on a single board and return the predicted move.
    If enforce_legal=True, pick the highest-logit legal move.
    """
    with torch.no_grad():
        logits = model(board_tokens.unsqueeze(0).to(DEVICE))  # (1, 4096)
        if enforce_legal and board is not None:
            # Mask illegal moves
            legal_indices = [encode_move(m) for m in board.legal_moves]
            mask = torch.full((4096,), float('-inf'))
            mask[legal_indices] = 0.0
            logits = logits + mask.to(DEVICE)
        move_idx = logits.argmax(dim=1).item()
    return decode_move(move_idx)


# Evaluate on full validation set
correct   = 0
legal     = 0
total     = 0

# Re-parse a small portion for board objects (needed for legality check)
# We evaluate on 10k random val positions
EVAL_POSITIONS = 10_000

# Build a small eval set with actual Board objects
eval_boards_raw = []
eval_moves_raw  = []

random_val_indices = random.sample(range(len(val_ds)), min(EVAL_POSITIONS, len(val_ds)))

for i in random_val_indices:
    b_tokens, m_idx = val_ds[i]
    eval_boards_raw.append(b_tokens)
    eval_moves_raw.append(m_idx.item())

with torch.no_grad():
    for b_tokens, true_move_idx in tqdm(zip(eval_boards_raw, eval_moves_raw),
                                         total=len(eval_boards_raw),
                                         desc='Evaluating'):
        logits = model(b_tokens.unsqueeze(0).to(DEVICE))  # (1, 4096)
        pred_idx = logits.argmax(dim=1).item()
        pred_move = decode_move(pred_idx)

        if pred_idx == true_move_idx:
            correct += 1

        # Reconstruct board from tokens is lossy, so we
        # do a lightweight legality check via python-chess
        # using the encoded tokens to rebuild a FEN isn't trivial,
        # so here we use a proxy: is the from_square occupied?
        piece_at_from = b_tokens[pred_move.from_square].item()
        if piece_at_from != 0:  # non-empty = likely legal-ish
            legal += 1

        total += 1

move_accuracy  = correct / total * 100
legal_move_rate = legal  / total * 100

print(f'Evaluated on : {total:,} positions')
print(f'Move Accuracy  : {move_accuracy:.2f}%')
print(f'Legal Move Rate (proxy): {legal_move_rate:.2f}%')

In [ ]:
# -------------------------------------------------------------------
# Metric 3 & 4: Top-K Match Rate + Centipawn Loss
# Requires Stockfish. Run on a smaller set (1000 positions) — slow
# -------------------------------------------------------------------

# Find stockfish binary
import shutil
_sf_candidates = [shutil.which('stockfish'), '/usr/games/stockfish',
                  '/usr/local/bin/stockfish']
SF_PATH = next((p for p in _sf_candidates if p and os.path.isfile(p)), None)
if SF_PATH is None:
    print('WARNING: Stockfish binary not found. '
          'Skipping Top-K and centipawn-loss evaluation.'
          ' Set topk_rate and avg_cp_loss to None.')
    topk_rate = None
    avg_cp_loss = None
    sf_samples = []
else:
    print(f'Stockfish path: {SF_PATH}')

    K = 5          # Top-K for match rate
    SF_DEPTH = 12  # Stockfish search depth (higher = stronger, slower)
    SF_EVAL_N = 1000  # Number of positions to evaluate with Stockfish

    # We need actual Board objects for Stockfish — re-parse a small PGN portion
    def sample_board_objects(pgn_path: str, n: int = 1000, min_elo: int = 1500):
        """Return n (Board, ground_truth_move) pairs from the PGN."""
        samples = []
        with open_pgn(pgn_path) as f:
            while len(samples) < n * 5:  # overshoot, filter later
                game = chess.pgn.read_game(f)
                if game is None:
                    break
                try:
                    we = int(game.headers.get('WhiteElo', 0))
                    be = int(game.headers.get('BlackElo', 0))
                    if we < min_elo or be < min_elo:
                        continue
                except (ValueError, TypeError):
                    continue

                board = game.board()
                moves = list(game.mainline_moves())
                # Sample a random move from the middle of the game
                if len(moves) < 10:
                    continue
                idx = random.randint(5, len(moves) - 5)
                for m in moves[:idx]:
                    board.push(m)
                samples.append((board.copy(), moves[idx]))
                if len(samples) >= n:
                    break
        return samples[:n]

    print('Sampling board positions for Stockfish evaluation...')
    sf_samples = sample_board_objects(PGN_FILE, n=SF_EVAL_N, min_elo=1500)
    print(f'Got {len(sf_samples)} positions.')


In [ ]:
if not sf_samples:
    print('No Stockfish samples available — skipping evaluation.')
else:
    topk_matches    = 0
    centipawn_losses = []
    illegal_moves   = 0

    engine = chess.engine.SimpleEngine.popen_uci(SF_PATH)

    for board, gt_move in tqdm(sf_samples, desc='Stockfish eval'):
        # Model prediction
        b_tokens = torch.tensor(encode_board(board), dtype=torch.long)
        with torch.no_grad():
            logits = model(b_tokens.unsqueeze(0).to(DEVICE))  # (1, 4096)

        # Get top-1 legal model move
        sorted_indices = logits.squeeze(0).argsort(descending=True).tolist()
        model_move = None
        for idx in sorted_indices:
            candidate = decode_move(idx)
            if candidate in board.legal_moves:
                model_move = candidate
                break

        if model_move is None:
            illegal_moves += 1
            continue

        # Stockfish analysis: get top-K moves and their scores
        try:
            info = engine.analyse(board,
                                  chess.engine.Limit(depth=SF_DEPTH),
                                  multipv=K)
        except Exception:
            continue

        # Top-K match
        top_k_moves = [pv['pv'][0] for pv in info if 'pv' in pv and pv['pv']]
        if model_move in top_k_moves:
            topk_matches += 1

        # Centipawn loss from the perspective of the player to move.
        # Using .pov(current_turn) keeps the sign correct for both colors:
        #   White to move: higher is better → best_pov >= model_pov when model is suboptimal
        #   Black to move: higher is better (for black) → same invariant holds
        current_turn = board.turn
        best_pov_cp = info[0]['score'].pov(current_turn).score(mate_score=10000)
        if best_pov_cp is None:
            continue

        # Evaluate model move with Stockfish
        board.push(model_move)
        try:
            model_info = engine.analyse(board, chess.engine.Limit(depth=SF_DEPTH))
            # After the push the opponent is to move; pov(current_turn) flips
            # the sign so we still measure from the mover's perspective.
            model_pov_cp = model_info['score'].pov(current_turn).score(mate_score=10000)
        except Exception:
            board.pop()
            continue
        board.pop()

        if model_pov_cp is not None:
            cp_loss = best_pov_cp - model_pov_cp
            centipawn_losses.append(max(0, cp_loss))

    engine.quit()

    n_eval = len(sf_samples) - illegal_moves
    topk_rate = topk_matches / n_eval * 100 if n_eval > 0 else 0
    avg_cp_loss = np.mean(centipawn_losses) if centipawn_losses else float('nan')

    print(f'\n===== Stockfish Evaluation Results =====')
    print(f'Positions evaluated : {n_eval}')
    print(f'Illegal move rate   : {illegal_moves / len(sf_samples) * 100:.1f}%')
    print(f'Top-{K} Match Rate   : {topk_rate:.1f}%')
    print(f'Avg Centipawn Loss  : {avg_cp_loss:.1f}')
    print(f'========================================')


In [ ]:
# ---- Plot centipawn loss distribution ----
fig, axes = plt.subplots(1, 2, figsize=(13, 4))

# Histogram of centipawn losses
axes[0].hist(centipawn_losses, bins=50, color='steelblue', edgecolor='white')
axes[0].axvline(avg_cp_loss, color='red', linestyle='--', label=f'Mean = {avg_cp_loss:.1f}')
axes[0].set_title('Centipawn Loss Distribution')
axes[0].set_xlabel('Centipawn Loss')
axes[0].set_ylabel('Count')
axes[0].legend()
axes[0].grid(True, alpha=0.3)

# Summary bar chart
metrics = {
    'Move\nAccuracy (%)': move_accuracy,
    f'Top-{K}\nMatch Rate (%)': topk_rate,
    'Legal\nMove Rate (%)': legal_move_rate,
}
bars = axes[1].bar(metrics.keys(), metrics.values(),
                   color=['#4c72b0', '#55a868', '#c44e52'], width=0.5)
axes[1].set_ylim(0, 100)
axes[1].set_title('Evaluation Summary')
axes[1].set_ylabel('%')
for bar, val in zip(bars, metrics.values()):
    axes[1].text(bar.get_x() + bar.get_width()/2, bar.get_height() + 1,
                 f'{val:.1f}%', ha='center', fontsize=11)
axes[1].grid(True, axis='y', alpha=0.3)

plt.tight_layout()
plt.savefig('evaluation_results.png', dpi=150)
plt.show()

## 8. Live Demo: Generate a Full Game

Let the model play both sides (White and Black) from the starting position. Visualize the game move by move.

In [ ]:
def model_vs_model(max_moves: int = 80, enforce_legal: bool = True):
    """
    Let the model play both sides from the starting position.
    Returns the final board and list of moves.
    """
    board  = chess.Board()
    moves  = []

    for _ in range(max_moves):
        if board.is_game_over():
            break

        b_tokens = torch.tensor(encode_board(board), dtype=torch.long)
        with torch.no_grad():
            logits = model(b_tokens.unsqueeze(0).to(DEVICE)).squeeze(0)

        if enforce_legal:
            # Pick the highest-logit legal move
            sorted_indices = logits.argsort(descending=True).tolist()
            chosen_move = None
            for idx in sorted_indices:
                candidate = decode_move(idx)
                if candidate in board.legal_moves:
                    chosen_move = candidate
                    break
            if chosen_move is None:
                break
        else:
            chosen_move = decode_move(logits.argmax().item())

        moves.append(chosen_move)
        board.push(chosen_move)

    return board, moves


final_board, game_moves = model_vs_model(max_moves=60)

print(f'Game length: {len(game_moves)} moves')
print(f'Result: {final_board.result()}')
print()
print('Move list:')
board_tmp = chess.Board()
move_strings = []
for i, move in enumerate(game_moves):
    san = board_tmp.san(move)
    move_strings.append(san)
    board_tmp.push(move)
    if i % 2 == 0:
        print(f'{i//2 + 1}. {san}', end=' ')
    else:
        print(san)

print(f'\nFinal position (FEN):\n{final_board.fen()}')

In [ ]:
# ---- Visualize a few positions from the game ----
import chess.svg
from IPython.display import display, SVG

board_tmp = chess.Board()

# Show board at move 1, 10, 20, and final
snapshots_at = [0, 10, 20, len(game_moves) - 1]
snapshots = []

for i, move in enumerate(game_moves):
    if i in snapshots_at:
        snapshots.append((f'After move {i+1}: {board_tmp.san(move)}',
                          board_tmp.copy()))
    board_tmp.push(move)

if len(game_moves) - 1 in snapshots_at and len(snapshots) == len(snapshots_at):
    snapshots[-1] = (f'Final position (move {len(game_moves)})', board_tmp.copy())

for title, b in snapshots:
    print(title)
    svg = chess.svg.board(b, size=300)
    display(SVG(svg))
    print()

## 9. Results Summary

In [ ]:
print('=' * 50)
print('      CHESS TRANSFORMER — RESULTS SUMMARY')
print('=' * 50)
print(f'  Model parameters    : {total_params:,}')
print(f'  Training positions  : {train_size:,}')
print(f'  Validation positions: {val_size:,}')
print()
if history['train_loss']:
    print(f'  Final Train Loss    : {history["train_loss"][-1]:.4f}')
    print(f'  Final Val Loss      : {history["val_loss"][-1]:.4f}')
    print(f'  Final Val Accuracy  : {history["val_acc"][-1]*100:.2f}%')
else:
    print('  (training history not available)')
print()
print(f'  Move Accuracy       : {move_accuracy:.2f}%')
print(f'  Legal Move Rate     : {legal_move_rate:.2f}%')
print(f'  Top-{K} Match Rate   : {topk_rate:.1f}%')
print(f'  Avg Centipawn Loss  : {avg_cp_loss:.1f} cp')
print()
print('  Reference centipawn losses:')
print('    Grandmaster  : ~20 cp')
print('    Club player  : ~50-80 cp')
print('    Random legal : ~300+ cp')
print('=' * 50)


## 10. Challenges and Limitations

- **No game context**: the model sees only the current board, not the sequence of prior moves. It cannot detect repetition or evaluate long-term plans.
- **Promotion handling**: moves with pawn promotion (e.g. `e7e8q`) are encoded without the promotion piece, so promotions may be predicted as non-promotions.
- **Castling and en passant**: these are legal moves that require additional board state (castling rights, en passant square). Our encoding does not include this information explicitly.
- **Dataset quality**: training on games of all Elos teaches the model bad habits. Filtering to 2000+ Elo games significantly improves move quality.
- **Scale**: professional chess engines (AlphaZero, Leela Chess Zero) use orders of magnitude more data and compute. This project demonstrates the core idea at classroom scale.